<a href="https://colab.research.google.com/github/UnfoldDataScience/Agentic_Ai_For_Beginner/blob/main/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

In [1]:
!python --version

Python 3.13.15


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   TAVILY_API_KEY

In [2]:
!pip install -q \
langchain==0.3.14 \
langchain-openai==0.2.14 \
langchain-community==0.3.14 \
openai==1.59.6 \
python-dotenv==1.0.1 \
requests==2.32.3 \
beautifulsoup4==4.12.3 \
wikipedia==1.4.0 \
ipykernel==6.29.5 \
tavily-python==0.5.0


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [2]:
import requests
from bs4 import BeautifulSoup

In [4]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

Hello, David! How can I help you today?


In [7]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

The capital of the United States is Washington, D.C.


In [8]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

Certainly! Here’s a list of important topics in Deep Learning, along with explanations of why each is significant, especially in light of recent advancements in the field of AI:

### 1. **Neural Network Architectures**
   - **Importance**: Understanding various neural network architectures (e.g., CNNs, RNNs, Transformers) is crucial as different architectures excel in different applications, such as image recognition, natural language processing, and time-series analysis. Recent advancements like Vision Transformers (ViTs) and advanced CNNs have pushed the boundaries of performance in multiple tasks.

### 2. **Transfer Learning**
   - **Importance**: With the advent of large pretrained models like BERT and GPT, transfer learning has become a powerful technique. It allows practitioners to leverage existing models, fine-tuning them for specific tasks with limited data, which is particularly important in resource-constrained environments.

### 3. **Generative Models**
   - **Importance**:

In [9]:
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple custom tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create tool from function
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)

# Tavily Search Tool
tavily_search = TavilySearchResults(max_results=2)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Tools list
tools = [tavily_search, my_tool]

# Create agent
agent = initialize_agent(
    tools,
    llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("What's the weather like today in London?")

print(response)



> Entering new AgentExecutor chain...
I need to find the current weather conditions in London. Since this is a current event, I will use the search tool to get the most accurate information.  
Action: tavily_search_results_json  
Action Input: "current weather in London"  
Observation: [{'url': 'https://timesofindia.indiatimes.com/world/uk/london-weather-forecast-sunny-intervals-and-mild-temperatures-today-august-24-2026/amp_articleshow/133451758.cms', 'content': "Comments\n\nShare\n\nRead Ad-Free on App\n\n### Today's Weather - August 24, 2026\n\n  \nLondon residents can expect sunny intervals and mild temperatures on August 24, 2026, with a maximum temperature of 22°C and a minimum of 16°C, making it a good day for outdoor activities. However, a slight chance of showers is anticipated later in the week, suggesting the need for an umbrella.  \n\nTired of too many ads?go ad free now [...] Tired of too many ads?go ad free now\n\nToday, August 24, 2026, London will experience sunny int

In [10]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [11]:
tools = [tavily_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [12]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])



> Entering new AgentExecutor chain...
I need to gather information about the inventor of the World Wide Web and its impact. This requires a search for reliable sources or summaries of relevant information. 

Action: tavily_search_results_json
Action Input: "Who invented the World Wide Web and what impact did it have?"

Observation: [{'url': 'https://en.wikipedia.org/wiki/World_Wide_Web', 'content': 'The Web was invented by English computer scientist Tim Berners-Lee while at CERN in 1989 and opened to the public in 1993. It was conceived as a "universal linked information system". Documents and other media content are made available to the network through web servers and can be accessed by programs such as web browsers. Servers and resources on the World Wide Web are identified and located through a character string called a uniform resource locator (URL). [...] The Web was invented by English computer scientist Tim Berners-Lee while working at CERN. He was motivated by the problem of